# Mổ xẻ GPT mini tiếng Việt của bạn (portfolio)

Bạn đã train một GPT tiếng Việt ở module 05. Giờ mở nó ra bằng mọi công cụ của module này và viết một báo cáo interpretability mà
một reviewer khó tính chấp nhận: mỗi kết luận có thí nghiệm, đối chứng, khoảng tin cậy, và một câu "điều này **không** chứng minh".

**Cách làm bài**
- Hai hàm được test chấm (cell dưới). Phần còn lại tự thiết kế; khung gợi ý ở mục 2.
- Checkpoint: đặt biến môi trường `GPT_MINI_CKPT` tới tệp `gpt-mini-v1` của bạn (module 05, bài Advanced 01). Cách khác ở README của bài.
- Dùng lại công cụ của các bài Basic và Intermediate (`hooks.py`, `lens.py`, `probe.py`, hàm patching, SAE).
- Chấm: `uv run pytest tracks/ai-engineer/modules/08-interpretability/exercises/advanced/01-interpret-gpt-mini -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm checkpoint
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
CKPT = os.environ.get("GPT_MINI_CKPT")
print("checkpoint:", CKPT, Path(CKPT).exists() if CKPT else "chưa đặt GPT_MINI_CKPT")

In [ ]:
import numpy as np
import torch

## 1. Hai hàm được chấm

- `top_examples`: lấy ví dụ kích hoạt mạnh nhất của một latent, mỗi ngữ cảnh tối đa một ví dụ để một văn bản dài không chiếm hết
  danh sách.
- `bootstrap_ci`: khoảng tin cậy cho mọi con số trung bình bạn báo cáo (hiệu ứng patching, accuracy probe theo ngữ cảnh).

In [ ]:
def top_examples(acts, k, row_len, one_per_row=True):
    """`acts` là tensor 1 chiều: kích hoạt của một latent (hoặc neuron, hướng probe) trên các token xếp theo ngữ cảnh rồi theo vị
    trí, mỗi ngữ cảnh `row_len` token. Trả list tuple `(row, pos, value)` (int, int, float) của tối đa k kích hoạt **dương** lớn
    nhất, giảm dần theo value; hoà thì chỉ số phẳng nhỏ hơn đứng trước. `one_per_row=True`: mỗi ngữ cảnh góp tối đa một ví dụ
    (vị trí có kích hoạt lớn nhất của ngữ cảnh đó).
    """
    # TODO
    raise NotImplementedError

In [ ]:
def bootstrap_ci(values, n_boot=1000, alpha=0.05, seed=0):
    """Khoảng tin cậy percentile bootstrap cho **trung bình** của `values` (mảng 1 chiều, ví dụ hiệu ứng patching từng prompt,
    hoặc đúng/sai của probe từng ngữ cảnh). Chỉ số mẫu lại: `np.random.default_rng(seed).integers(0, n, size=(n_boot, n))`.
    Trả `(lo, hi)` là `np.quantile` mức `alpha/2` và `1 - alpha/2` của n_boot trung bình, kiểu float.
    """
    # TODO
    raise NotImplementedError

## 2. Khung gợi ý (không chấm)

1. **Nạp model**: `gpt-mini-v1` (khoá `format`, `config`, `state_dict`, `tokenizer`) vào lớp `GPT` của module 05. Văn bản đánh giá:
   validation Wikipedia của module 04/05 hoặc `tst2012.vi` + `tst2013.vi` của IWSLT15.
2. **Bản đồ head**: previous-token, duplicate, induction; ablation (mean) head induction so với head ngẫu nhiên.
3. **Logit lens**: CE và trùng top-1 theo tầng; so với GPT-2 small ở bài Intermediate 01.
4. **Probe có đối chứng**: một thuộc tính tiếng Việt bạn chọn (ví dụ token đã xuất hiện, token bắt đầu một từ ghép, vị trí trong
   câu, cảm xúc câu của UIT-VSFC trung bình theo câu); control task, model khởi tạo ngẫu nhiên, baseline theo loại token.
5. **Patching**: một nhiệm vụ có cặp clean/corrupted khác nhau một token mà model của bạn làm được; thước đo logit difference;
   bảng residual theo (tầng, vị trí) và theo head; khoảng tin cậy bootstrap theo cặp prompt.
6. **SAE**: một tầng giữa, TopK; FVU, L0, latent chết, CE recovered với cả hai baseline; đọc ít nhất 10 latent bằng `top_examples`.
7. Viết `REPORT.md`.

In [ ]:
# explore: TODO: pipeline của bạn